# Newsletter Daily Ingestion Pipeline

This notebook:
1. Ingests RSS feeds from Databricks documentation
2. Fetches page content
3. Runs key information extraction

**Data Flow:**
- RSS feed → `feed_items` table (Lakebase)
- Page URLs → `page_content` table (Lakebase)
- Content → `kie_outputs` table (Lakebase)

## 1. Install Dependencies

In [ ]:
%pip install -q feedparser beautifulsoup4 psycopg2-binary lxml python-dotenv databricks-sdk
dbutils.library.restartPython()

## 2. Setup Parameters

In [ ]:
# Get job parameters with defaults
dbutils.widgets.text("max_items", "100", "Max items to ingest")
dbutils.widgets.dropdown("run_mode", "production", ["production", "test"], "Run mode")

# Lakebase configuration parameters (set by DAB)
dbutils.widgets.text("lakebase_endpoint", "", "Lakebase endpoint path")
dbutils.widgets.text("pghost", "", "PostgreSQL hostname")
dbutils.widgets.text("pgdatabase", "databricks_postgres", "PostgreSQL database")

max_items = int(dbutils.widgets.get("max_items"))
run_mode = dbutils.widgets.get("run_mode")

print(f"📊 Run Mode: {run_mode}")
print(f"📊 Max Items: {max_items}")

## 3. Configure Lakebase Connection

**Two authentication options:**
- **Option A**: SDK-generated OAuth tokens (recommended, no secrets needed)
- **Option B**: Native Postgres password from Databricks Secrets (simpler, works now)

In [ ]:
import os
import sys

# Add src/ to Python path so we can import our modules
sys.path.insert(0, '/Workspace' + dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().rsplit('/', 1)[0] + '/../src')

# OAuth Token Authentication (RECOMMENDED - No secrets needed!)
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Read configuration from notebook parameters (set by DAB)
lakebase_endpoint = dbutils.widgets.get("lakebase_endpoint")
pghost = dbutils.widgets.get("pghost")
pgdatabase = dbutils.widgets.get("pgdatabase")

# Get current user for PGUSER (uses Databricks identity)
current_user = w.current_user.me()
pguser = current_user.user_name  # Your Databricks email/username

if not lakebase_endpoint:
    raise ValueError("lakebase_endpoint parameter not set. Configure in databricks.yml variables section")

if not pghost:
    raise ValueError("pghost parameter not set. Configure in databricks.yml variables section")

# Generate fresh OAuth token for this run
print(f"🔐 Generating OAuth token...")
print(f"   Endpoint: {lakebase_endpoint}")
cred = w.postgres.generate_database_credential(endpoint=lakebase_endpoint)

# Configure Lakebase connection
os.environ['PGHOST'] = pghost
os.environ['PGDATABASE'] = pgdatabase
os.environ['PGUSER'] = pguser
os.environ['PGPASSWORD'] = cred.token  # Generated OAuth token
os.environ['PGSSLMODE'] = 'require'

print(f"✅ OAuth token generated successfully")
print(f"   Database: {pghost}/{pgdatabase}")
print(f"   User: {pguser}")
print(f"   Token length: {len(cred.token)} chars")
print(f"   Expires: ~1 hour from now")

## 4. Run Ingestion Pipeline

In [ ]:
from schema import get_connection
from ingest import ingest_all_sources

print("="*70)
print("STEP 1: RSS FEED INGESTION")
print("="*70)

conn = get_connection()  # Auto-detects Lakebase from PGHOST env var
db_type = "LAKEBASE (PostgreSQL)" if os.getenv('PGHOST') else "SQLite"
print(f"✅ Connected to: {db_type}\n")

# Ingest RSS feeds
stats = ingest_all_sources(conn, max_items=max_items if run_mode == "production" else 10)

total_new = sum(s['new_items'] for s in stats)
total_updated = sum(s['updated_items'] for s in stats)

print(f"\n✅ Ingested {total_new} new items, updated {total_updated} items")

# Track metrics for job monitoring
dbutils.jobs.taskValues.set(key="items_ingested", value=total_new)
dbutils.jobs.taskValues.set(key="items_updated", value=total_updated)

## 5. Fetch Page Content

In [ ]:
from fetch import fetch_items_content

print("="*70)
print("STEP 2: PAGE CONTENT FETCHING")
print("="*70)

fetch_stats = fetch_items_content(
    conn,
    skip_existing=True,
    delay=0.5  # Rate limiting
)

print(f"\n✅ Fetched {fetch_stats['fetched']} pages")
print(f"⚠️  Failed {fetch_stats['failed']} pages")

# Alert if failure rate is too high
if fetch_stats['total'] > 0:
    failure_rate = fetch_stats['failed'] / fetch_stats['total']
    if failure_rate > 0.3:
        raise Exception(f"High failure rate: {failure_rate:.1%} ({fetch_stats['failed']}/{fetch_stats['total']})")

# Track metrics
dbutils.jobs.taskValues.set(key="pages_fetched", value=fetch_stats['fetched'])
dbutils.jobs.taskValues.set(key="pages_failed", value=fetch_stats['failed'])

## 6. Key Information Extraction

In [ ]:
from kie import run_kie_extraction

print("="*70)
print("STEP 3: KEY INFORMATION EXTRACTION")
print("="*70)

kie_stats = run_kie_extraction(
    conn,
    skip_existing=True,
    model_version="stub-v1"  # TODO: Update when Agent Bricks is integrated
)

print(f"\n✅ Processed {kie_stats['processed']} items")
if kie_stats['errors']:
    print(f"⚠️  Errors: {len(kie_stats['errors'])}")
    for error in kie_stats['errors'][:5]:  # Show first 5
        print(f"   - {error}")

# Track metrics
dbutils.jobs.taskValues.set(key="kie_processed", value=kie_stats['processed'])
dbutils.jobs.taskValues.set(key="kie_errors", value=len(kie_stats['errors']))

## 7. Summary & Cleanup

In [ ]:
print("="*70)
print("PIPELINE SUMMARY")
print("="*70)
print(f"📊 Items ingested:  {total_new}")
print(f"📊 Pages fetched:   {fetch_stats['fetched']}")
print(f"📊 KIE processed:   {kie_stats['processed']}")
print(f"📊 Total errors:    {fetch_stats['failed'] + len(kie_stats['errors'])}")

conn.close()

print("\n✅ Pipeline completed successfully!")

## Next Steps

1. **Integrate Agent Bricks** - Replace KIE stub with real extraction
2. **Add weekly newsletter job** - Triggered by table updates
3. **Setup monitoring** - Alerts via email/Slack
4. **Add quality gates** - Pre-publish validation